In [1]:
# pip install pandas requests tqdm pymupdf
import json, re, os, requests, fitz, pandas as pd
from tqdm import tqdm

In [3]:
META = "data/arxiv-metadata-oai-snapshot.json"
OUT  = "data/arxiv_csml_body_abstract.csv"
MAX_DOCS = 200  # stop after this many papers (set higher for full run)

In [2]:
df = pd.read_csv("data/arxiv_csml_body_abstract.csv")

In [7]:
df.shape

(200, 4)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   paper_id  200 non-null    float64
 1   title     200 non-null    object 
 2   abstract  200 non-null    object 
 3   body      200 non-null    object 
dtypes: float64(1), object(3)
memory usage: 6.4+ KB


In [6]:
df.head()

,paper_id,title,abstract,body
0,704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-pe...",Sparsity-certifying Graph Decompositions Ilean...
1,704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is describe...,The evolution of the Earth-Moon system based o...
2,704.0021,Molecular Synchronization Waves in Arrays of A...,Spatiotemporal pattern formation in a product-...,arXiv:0704.0021v2 [nlin.PS] 24 Jul 2007 Molecu...
3,704.0033,Convergence of the discrete dipole approximati...,We performed a rigorous theoretical convergenc...,1 Convergence of the Discrete Dipole Approxima...
4,704.0035,Convergence of the discrete dipole approximati...,We propose an extrapolation technique that all...,1 Convergence of the discrete dipole approxima...


In [4]:
def is_csml(cats):
    s = (cats or "").lower()
    return ("cs." in s) or ("stat.ml" in s)

def pdf_to_text(fp):
    try:
        doc = fitz.open(fp)
        txt = "\n".join(page.get_text() for page in doc)
        doc.close()
        return re.sub(r"\s+", " ", txt).strip()
    except:
        return ""

os.makedirs("data/pdfs", exist_ok=True)
rows = []

with open(META, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Scanning metadata"):
        rec = json.loads(line)
        cats = rec.get("categories","")
        if not is_csml(cats):
            continue
        arxiv_id = rec.get("id","").strip()
        title = (rec.get("title") or "").strip()
        abstract = re.sub(r"\s+"," ", (rec.get("abstract") or "").strip())
        if len(abstract) < 50:
            continue
        
        pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
        pdf_fp = f"data/pdfs/{arxiv_id.replace('/','_')}.pdf"
        if not os.path.exists(pdf_fp):
            r = requests.get(pdf_url, timeout=60)
            if r.status_code == 200:
                with open(pdf_fp,"wb") as w: w.write(r.content)
            else:
                continue
        
        body = pdf_to_text(pdf_fp)
        if len(body) < 500:
            continue

        rows.append((arxiv_id, title, abstract, body))
        if len(rows) >= MAX_DOCS:
            break

FileNotFoundError: [Errno 2] No such file or directory: 'data/arxiv-metadata-oai-snapshot.json'

In [7]:
df = pd.DataFrame(rows, columns=["paper_id","title","abstract","body"])
df.to_csv(OUT, index=False, encoding="utf-8")
print(f"Saved {len(df)} rows -> {OUT}")

Saved 200 rows -> data/arxiv_csml_body_abstract.csv


In [11]:
df.shape

(200, 4)

In [8]:
df.head()

,paper_id,title,abstract,body
0,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-pe...",Sparsity-certifying Graph Decompositions Ilean...
1,0704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is describe...,The evolution of the Earth-Moon system based o...
2,0704.0021,Molecular Synchronization Waves in Arrays of A...,Spatiotemporal pattern formation in a product-...,arXiv:0704.0021v2 [nlin.PS] 24 Jul 2007 Molecu...
3,0704.0033,Convergence of the discrete dipole approximati...,We performed a rigorous theoretical convergenc...,1 Convergence of the Discrete Dipole Approxima...
4,0704.0035,Convergence of the discrete dipole approximati...,We propose an extrapolation technique that all...,1 Convergence of the discrete dipole approxima...


In [9]:
df.tail()

,paper_id,title,abstract,body
195,0704.1589,Improving immunization strategies,We introduce an immunization method where the ...,arXiv:0704.1589v1 [physics.soc-ph] 12 Apr 2007...
196,0704.1602,What does Hirsch index evolution explain us? A...,The evolution of Turkish Journal of Chemistry ...,"Año 8, No.27, Ene – Mar. 2007 1 What does Hirs..."
197,0704.1610,High-altitude signatures of ionospheric densit...,We present Cluster measurements of large elect...,1 High-altitude signatures of ionospheric dens...
198,0704.1622,MATLAB codes for teaching quantum physics: Part 1,Among the ideas to be conveyed to students in ...,arXiv:0704.1622v1 [physics.ed-ph] 12 Apr 2007 ...
199,0704.1627,Thin elastic shells with variable thickness fo...,Planetary topography can either be modeled as ...,arXiv:0704.1627v2 [physics.geo-ph] 14 Sep 2007...


In [10]:
df.head()['abstract']

0    We describe a new algorithm, the $(k,\ell)$-pe...
1    The evolution of Earth-Moon system is describe...
2    Spatiotemporal pattern formation in a product-...
3    We performed a rigorous theoretical convergenc...
4    We propose an extrapolation technique that all...
Name: abstract, dtype: object